# 03 — Train Input + Output Classifiers

Fit the input classifier (TF-IDF + logistic regression on canonical text) and the output classifier on the leakage corpus. Save artifacts under `../models/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

PROC = Path('../data/processed')
MODELS = Path('../models'); MODELS.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')

In [ ]:
train = pd.read_parquet(PROC / 'prompts_train.parquet')
redteam = pd.read_parquet(PROC / 'redteam.parquet')
outputs = pd.read_parquet(PROC / 'outputs.parquet')
print('train:', len(train), '  redteam:', len(redteam), '  outputs:', len(outputs))

## Train input classifier

In [ ]:
from prompt_guard.models import train_input_classifier
input_clf = train_input_classifier(train)
joblib.dump(input_clf, MODELS / 'input_clf.pkl')
print('input classifier trained on', len(train), 'rows')

## Evaluate on the held-out red-team set

In [ ]:
from prompt_guard.models import evaluate_input
eval_05 = evaluate_input(input_clf, redteam, threshold=0.5)
print(
    f"recall={eval_05['recall_injection']:.3f}  f1={eval_05['f1_injection']:.3f}  fpr={eval_05['fpr_benign']:.3f}"
)
print('per-type:', eval_05['per_type_recall'])

## Threshold sweep

In [ ]:
thresholds = np.linspace(0.1, 0.9, 17)
rows = []
for t in thresholds:
    e = evaluate_input(input_clf, redteam, threshold=float(t))
    rows.append({'threshold': float(t), 'recall': e['recall_injection'], 'fpr': e['fpr_benign']})
df = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(7,3))
ax.plot(df['threshold'], df['recall'], label='recall', marker='o')
ax.plot(df['threshold'], df['fpr'], label='fpr', marker='x')
ax.set_xlabel('threshold')
ax.legend()
ax.set_title('Recall vs FPR across thresholds')
plt.show()

## Per-type recall

In [ ]:
by_type = pd.Series(eval_05['per_type_recall'])
fig, ax = plt.subplots(figsize=(7,3))
by_type.plot(kind='barh', ax=ax, color='teal')
ax.set_xlim(0,1)
ax.set_title('Per-type recall on the red-team set')
for i, v in enumerate(by_type.values):
    ax.text(v + 0.01, i, f'{v:.2f}', va='center')
plt.show()

## Train output classifier

In [ ]:
from prompt_guard.models import train_output_classifier
output_clf = train_output_classifier(outputs)
joblib.dump(output_clf, MODELS / 'output_clf.pkl')
print('output classifier trained on', len(outputs), 'rows')

## Output classifier eval (held-out 1k)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
tr, te = train_test_split(outputs, test_size=0.2, stratify=outputs['is_leakage'], random_state=42)
from prompt_guard.models import train_output_classifier
model = train_output_classifier(tr)
preds = model.predict(te['text'].tolist())
print(classification_report(te['is_leakage'], preds))

## Confusion matrix — input classifier on red-team

In [ ]:
from prompt_guard.features import canonicalize
from sklearn.metrics import confusion_matrix
preds = input_clf.predict(redteam['text'].apply(canonicalize).tolist())
cm = confusion_matrix(redteam['is_injection'], preds)
fig, ax = plt.subplots(figsize=(4,3.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['benign','inj'], yticklabels=['benign','inj'], ax=ax)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title('Input classifier — red-team confusion')
plt.show()

## Save summary

In [ ]:
summary = {
    'recall_injection': eval_05['recall_injection'],
    'fpr_benign': eval_05['fpr_benign'],
    'f1_injection': eval_05['f1_injection'],
    'per_type_recall': eval_05['per_type_recall'],
}
summary